# Archived draft — superseded by ML_MODELS.ipynb

This notebook is an early development draft of the ConvLSTM / 3D-CNN
flood-nowcasting training pipeline, kept for reference only. It predates
the finalised `ML_MODELS.ipynb` and is **not used** by the current
pipeline. Identified as a draft because:

- Checkpoint paths use Unix/macOS-style absolute paths
  (`/Volumes/Seagate_2TB/...`), from before the project moved to this
  Windows machine's `W:\Dissertation\Volumes\Seagate_2TB\...` layout.
- Training reads directly from the CSV in per-hour chunks
  (`ChunkPatchDataset`, `BATCH_SIZE=4`, `N_EPOCHS=20`) — the slow
  approach later replaced by pre-converting to binary `.npy` arrays
  (10-20x faster), which is what `ML_MODELS.ipynb` actually uses.
- Model signatures are older/different: `CNN3DForecaster` here has
  `n_quantiles=9` and no `dropout` parameter, and `WeightedPinballLoss`
  uses an earlier 27:6:1 class-imbalance weighting scheme — both since
  superseded in `convlstm_model.py` / `cnn3d_model.py`.
- The same class definitions (`ConvLSTMForecaster`, `CNN3DForecaster`,
  `WeightedPinballLoss`) are pasted repeatedly across cells, typical of
  interactive iteration rather than a finished script.

This file was recovered from an unsaved VS Code "Untitled" tab and saved
under this name for archival purposes.

In [2]:
convlstm_code = '''
import torch
import torch.nn as nn
from tqdm import tqdm

class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size, padding=padding, bias=True)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch_size, height, width, device):
        return (
            torch.zeros(batch_size, self.hidden_channels, height, width, device=device),
            torch.zeros(batch_size, self.hidden_channels, height, width, device=device),
        )

class ConvLSTMLayer(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.cell = ConvLSTMCell(in_channels, hidden_channels, kernel_size)

    def forward(self, x_seq, pbar=None):
        B, T, C, H, W = x_seq.shape
        device = x_seq.device
        h, c = self.cell.init_hidden(B, H, W, device)
        for t in range(T):
            h, c = self.cell(x_seq[:, t], h, c)
            if pbar is not None:
                pbar.update(1)
        return h

class ConvLSTMForecaster(nn.Module):
    def __init__(self, in_channels=21, hidden_dims=(64, 128, 64),
                 n_quantiles=9, kernel_size=3):
        super().__init__()
        self.n_quantiles = n_quantiles
        self.layer1 = ConvLSTMLayer(in_channels,    hidden_dims[0], kernel_size)
        self.layer2 = ConvLSTMLayer(hidden_dims[0], hidden_dims[1], kernel_size)
        self.layer3 = ConvLSTMLayer(hidden_dims[1], hidden_dims[2], kernel_size)
        self.output_head = nn.Sequential(
            nn.Conv2d(hidden_dims[2], 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, n_quantiles, kernel_size=1),
        )

    def forward(self, x_seq, sort_quantiles=False, pbar=None):
        T = x_seq.shape[1]
        h1     = self.layer1(x_seq, pbar=pbar)
        h1_seq = h1.unsqueeze(1).expand(-1, T, -1, -1, -1)
        h2     = self.layer2(h1_seq, pbar=pbar)
        h2_seq = h2.unsqueeze(1).expand(-1, T, -1, -1, -1)
        h3     = self.layer3(h2_seq, pbar=pbar)
        out    = self.output_head(h3)
        if sort_quantiles:
            out, _ = torch.sort(out, dim=1)
        return out

class WeightedPinballLoss(nn.Module):
    def __init__(self, quantile_levels=None,
                 weight_heavy=10.0, weight_avg=3.0, weight_none=1.0,
                 threshold_heavy=95.0, threshold_avg=80.0):
        super().__init__()
        if quantile_levels is None:
            quantile_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
        self.register_buffer("taus",
            torch.tensor(quantile_levels, dtype=torch.float32))
        self.weight_heavy    = weight_heavy
        self.weight_avg      = weight_avg
        self.weight_none     = weight_none
        self.threshold_heavy = threshold_heavy
        self.threshold_avg   = threshold_avg

    def forward(self, pred, target):
        valid_mask = ~torch.isnan(target)
        if valid_mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True, device=pred.device)
        weights = torch.full_like(target, self.weight_none)
        weights = torch.where(target >= self.threshold_avg,
                              torch.full_like(weights, self.weight_avg), weights)
        weights = torch.where(target >= self.threshold_heavy,
                              torch.full_like(weights, self.weight_heavy), weights)
        target_exp = target.unsqueeze(1)
        taus_exp   = self.taus.view(1, -1, 1, 1)
        error      = target_exp - pred
        loss       = torch.where(error >= 0,
                                 taus_exp * error,
                                 (taus_exp - 1) * error)
        loss_mean     = loss.mean(dim=1)
        loss_weighted = loss_mean * weights * valid_mask.float()
        return loss_weighted.sum() / valid_mask.float().sum()
'''

cnn3d_code = '''
import torch
import torch.nn as nn
from tqdm import tqdm
from convlstm_model import WeightedPinballLoss

class Conv3dBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=(3,3,3), stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size,
                      stride=stride, padding=padding, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class CNN3DForecaster(nn.Module):
    def __init__(self, in_channels=21, n_quantiles=9):
        super().__init__()
        self.n_quantiles = n_quantiles
        self.block1 = Conv3dBlock(in_channels, 32,
                                  kernel_size=(3,3,3), padding=(1,1,1))
        self.block2 = Conv3dBlock(32, 64,
                                  kernel_size=(3,3,3),
                                  stride=(2,1,1), padding=(1,1,1))
        self.block3 = Conv3dBlock(64, 128,
                                  kernel_size=(3,3,3),
                                  stride=(2,1,1), padding=(1,1,1))
        self.block4 = Conv3dBlock(128, 64,
                                  kernel_size=(3,3,3),
                                  stride=(3,1,1), padding=(1,1,1))
        self.output_head = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, n_quantiles, kernel_size=1),
        )

    def forward(self, x_seq, sort_quantiles=False, pbar=None):
        x = x_seq.permute(0, 2, 1, 3, 4)
        x = self.block1(x)
        if pbar is not None: pbar.update(1)
        x = self.block2(x)
        if pbar is not None: pbar.update(1)
        x = self.block3(x)
        if pbar is not None: pbar.update(1)
        x = self.block4(x)
        if pbar is not None: pbar.update(1)
        x   = x.squeeze(2)
        out = self.output_head(x)
        if sort_quantiles:
            out, _ = torch.sort(out, dim=1)
        return out
'''

# Write both files to THESIS_PROJECT directory
with open("convlstm_model.py", "w") as f:
    f.write(convlstm_code)
print("✓ Written convlstm_model.py")

with open("cnn3d_model.py", "w") as f:
    f.write(cnn3d_code)
print("✓ Written cnn3d_model.py")

✓ Written convlstm_model.py
✓ Written cnn3d_model.py


# CONVLSTM

In [1]:
# CONVLSTM MODEL — PROBABILISTIC UK FLOOD NOWCASTING
# ============================================================================
# Architecture: ConvLSTM encoder → quantile output head
#
# Input:  (B, T, C, H, W) = (B, 12, 21, 140, 131)
# Output: (B, 9, H, W)    = (B, 9,  140, 131)
#         9 quantile estimates (τ = 0.1, 0.2, ..., 0.9)
#
# Loss: Weighted pinball loss (≈ CRPS)
#       heavy flood ×10, average flood ×3, no flood ×1
# ============================================================================
import torch
import torch.nn as nn
from tqdm import tqdm


# ── ConvLSTM Cell ──────────────────────────────────────────────────────────

class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        padding   = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size, padding=padding, bias=True
        )

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates    = self.conv(combined)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch_size, height, width, device):
        return (
            torch.zeros(batch_size, self.hidden_channels,
                        height, width, device=device),
            torch.zeros(batch_size, self.hidden_channels,
                        height, width, device=device),
        )


# ── ConvLSTM Layer ─────────────────────────────────────────────────────────

class ConvLSTMLayer(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.cell = ConvLSTMCell(in_channels, hidden_channels, kernel_size)

    def forward(self, x_seq, pbar=None):
        """
        Args:
            x_seq : (B, T, C, H, W)
            pbar  : optional tqdm bar to update per timestep
        Returns:
            h : (B, hidden_channels, H, W) final hidden state
        """
        B, T, C, H, W = x_seq.shape
        device = x_seq.device
        h, c = self.cell.init_hidden(B, H, W, device)
        for t in range(T):
            h, c = self.cell(x_seq[:, t], h, c)
            if pbar is not None:
                pbar.update(1)
        return h


# ── Full ConvLSTM Model ────────────────────────────────────────────────────

class ConvLSTMForecaster(nn.Module):
    """
    Three-layer ConvLSTM encoder with 9-channel quantile output head.

    Layers:
        Layer 1: 21  → 64  channels
        Layer 2: 64  → 128 channels
        Layer 3: 128 → 64  channels
        Output:  64  → 9   channels (1×1 conv)
    """
    def __init__(self, in_channels=21, hidden_dims=(64, 128, 64),
                 n_quantiles=9, kernel_size=3):
        super().__init__()
        self.n_quantiles = n_quantiles
        self.layer1 = ConvLSTMLayer(in_channels,    hidden_dims[0], kernel_size)
        self.layer2 = ConvLSTMLayer(hidden_dims[0], hidden_dims[1], kernel_size)
        self.layer3 = ConvLSTMLayer(hidden_dims[1], hidden_dims[2], kernel_size)
        self.output_head = nn.Sequential(
            nn.Conv2d(hidden_dims[2], 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, n_quantiles, kernel_size=1),
        )

    def forward(self, x_seq, sort_quantiles=False, pbar=None):
        """
        Args:
            x_seq         : (B, T, C, H, W)
            sort_quantiles: sort outputs at inference to prevent crossing
            pbar          : optional tqdm bar (updated per timestep per layer)
        """
        T = x_seq.shape[1]

        h1     = self.layer1(x_seq, pbar=pbar)
        h1_seq = h1.unsqueeze(1).expand(-1, T, -1, -1, -1)
        h2     = self.layer2(h1_seq, pbar=pbar)
        h2_seq = h2.unsqueeze(1).expand(-1, T, -1, -1, -1)
        h3     = self.layer3(h2_seq, pbar=pbar)

        out = self.output_head(h3)

        if sort_quantiles:
            out, _ = torch.sort(out, dim=1)
        return out


# ── Weighted Pinball Loss ──────────────────────────────────────────────────

class WeightedPinballLoss(nn.Module):
    """
    Pinball loss averaged across 9 quantile levels (≈ CRPS).
    Sample weights address 27:6:1 class imbalance:
        heavy flood (≥ 95) : ×10
        average flood (80–95): ×3
        no flood (< 80)    : ×1
    """
    def __init__(self, quantile_levels=None,
                 weight_heavy=10.0, weight_avg=3.0, weight_none=1.0,
                 threshold_heavy=95.0, threshold_avg=80.0):
        super().__init__()
        if quantile_levels is None:
            quantile_levels = [0.1, 0.2, 0.3, 0.4, 0.5,
                                0.6, 0.7, 0.8, 0.9]
        self.register_buffer('taus',
            torch.tensor(quantile_levels, dtype=torch.float32))
        self.weight_heavy    = weight_heavy
        self.weight_avg      = weight_avg
        self.weight_none     = weight_none
        self.threshold_heavy = threshold_heavy
        self.threshold_avg   = threshold_avg

    def forward(self, pred, target):
        valid_mask = ~torch.isnan(target)
        if valid_mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True, device=pred.device)

    # Replace NaN targets with 0 for computation (masked out anyway)
        target_clean = target.clone()
        target_clean[~valid_mask] = 0.0

        weights = torch.full_like(target_clean, self.weight_none)
        weights = torch.where(target_clean >= self.threshold_avg,
                          torch.full_like(weights, self.weight_avg), weights)
        weights = torch.where(target_clean >= self.threshold_heavy,
                          torch.full_like(weights, self.weight_heavy), weights)

        target_exp = target_clean.unsqueeze(1)
        taus_exp   = self.taus.view(1, -1, 1, 1)
        error      = target_exp - pred
        loss       = torch.where(error >= 0,
                             taus_exp * error,
                             (taus_exp - 1) * error)
        loss_mean     = loss.mean(dim=1)
        loss_weighted = loss_mean * weights * valid_mask.float()
        return loss_weighted.sum() / valid_mask.float().sum()

# ── Sanity Check ──────────────────────────────────────────────────────────

if __name__ == "__main__":
    device = torch.device("cpu")
    B, T, C, H, W = 2, 12, 21, 140, 131

    model     = ConvLSTMForecaster(in_channels=21,
                                    hidden_dims=(64, 128, 64),
                                    n_quantiles=9).to(device)
    criterion = WeightedPinballLoss()

    x      = torch.randn(B, T, C, H, W)
    target = torch.rand(B, H, W) * 100         # guaranteed 0-100 range
    target[0, :5, :5] = float('nan')

    # Progress bar for sanity check forward pass
    total_steps = 3 * T  # 3 layers × T timesteps
    with tqdm(total=total_steps, desc="ConvLSTM forward pass",
              unit="step",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} timesteps "
                         "[{elapsed}<{remaining}]") as pbar:
        pred = model(x, sort_quantiles=False, pbar=pbar)

    loss = criterion(pred, target)

    print(f"\nInput shape:  {x.shape}")
    print(f"Output shape: {pred.shape}")
    print(f"Target shape: {target.shape}")
    print(f"Pinball loss: {loss.item():.4f}")
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters:   {n_params:,}")
    print("✓ ConvLSTM architecture check passed")

ConvLSTM forward pass: 100%|██████████| 36/36 timesteps [00:01<00:00]


Input shape:  torch.Size([2, 12, 21, 140, 131])
Output shape: torch.Size([2, 9, 140, 131])
Target shape: torch.Size([2, 140, 131])
Pinball loss: 60.6062
Parameters:   1,542,729
✓ ConvLSTM architecture check passed


In [2]:
import torch
import sys
sys.path.insert(0, "D:/")
from convlstm_model import ConvLSTMForecaster, WeightedPinballLoss
device = torch.device("cpu")
B, T, C, H, W = 2, 12, 21, 140, 131

model     = ConvLSTMForecaster().to(device)
criterion = WeightedPinballLoss()

x      = torch.randn(B, T, C, H, W)
target = torch.rand(B, H, W) * 100    # guaranteed 0-100, no negatives
target[0, :5, :5] = float('nan')      # sea cells

pred = model(x)
loss = criterion(pred, target)
print(f"Loss with valid target: {loss.item():.4f}")
print(f"Pred range: [{pred.min():.2f}, {pred.max():.2f}]")
print(f"Any NaN in pred: {pred.isnan().any()}")

Loss with valid target: nan
Pred range: [-0.11, 0.17]
Any NaN in pred: False


# 3D CNN

In [3]:
from cnn3d_model import CNN3DForecaster
from convlstm_model import WeightedPinballLoss
from tqdm import tqdm
import torch

device = torch.device("cpu")
B, T, C, H, W = 2, 12, 21, 140, 131

model     = CNN3DForecaster().to(device)
criterion = WeightedPinballLoss()

x      = torch.randn(B, T, C, H, W)
target = torch.rand(B, H, W) * 100
target[0, :5, :5] = float('nan')

with tqdm(total=4, desc="3D-CNN forward pass", unit="block",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} blocks "
                     "[{elapsed}<{remaining}]") as pbar:
    pred = model(x, sort_quantiles=False, pbar=pbar)

loss = criterion(pred, target)

print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {pred.shape}")
print(f"Target shape: {target.shape}")
print(f"Pinball loss: {loss.item():.4f}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters:   {n_params:,}")
print("✓ 3D-CNN architecture check passed")

3D-CNN forward pass: 100%|██████████| 4/4 blocks [00:00<00:00]


Input shape:  torch.Size([2, 12, 21, 140, 131])
Output shape: torch.Size([2, 9, 140, 131])
Target shape: torch.Size([2, 140, 131])
Pinball loss: nan
Parameters:   535,145
✓ 3D-CNN architecture check passed


In [4]:
# 3D-CNN MODEL — PROBABILISTIC UK FLOOD NOWCASTING
# ============================================================================
# Updated channel dimensions to match ConvLSTM parameter count (~1.5M):
#   Block 1: 21  → 64  channels
#   Block 2: 64  → 128 channels
#   Block 3: 128 → 256 channels
#   Block 4: 256 → 128 channels
# Fixed TypeError: kernel_size and stride now passed as explicit tuples
# throughout to avoid PyTorch unpacking errors.
# ============================================================================
import torch
import torch.nn as nn
from tqdm import tqdm


class Conv3dBlock(nn.Module):
    """3D Conv → BatchNorm3d → ReLU"""
    def __init__(self, in_ch, out_ch,
                 kernel_size=(3, 3, 3),
                 stride=(1, 1, 1),
                 padding=(1, 1, 1)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch,
                      kernel_size=kernel_size,
                      stride=stride,
                      padding=padding,
                      bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class CNN3DForecaster(nn.Module):
    """
    Four-layer 3D-CNN encoder with temporal collapse and
    9-channel quantile output head.

    Channel dimensions scaled to match ConvLSTM (~1.5M parameters):
        Block 1: (21,  12, H, W) → (64,  12, H, W)
        Block 2: (64,  12, H, W) → (128,  6, H, W)  stride=(2,1,1)
        Block 3: (128,  6, H, W) → (256,  3, H, W)  stride=(2,1,1)
        Block 4: (256,  3, H, W) → (128,  1, H, W)  stride=(3,1,1)
        Squeeze → (128, H, W)
        Output:  (128, H, W)    → (9,   H, W)
    """
    def __init__(self, in_channels=21, n_quantiles=9):
        super().__init__()
        self.n_quantiles = n_quantiles

        self.block1 = Conv3dBlock(in_channels, 64,
                                  kernel_size=(3, 3, 3),
                                  stride=(1, 1, 1),
                                  padding=(1, 1, 1))
        self.block2 = Conv3dBlock(64, 128,
                                  kernel_size=(3, 3, 3),
                                  stride=(2, 1, 1),
                                  padding=(1, 1, 1))
        self.block3 = Conv3dBlock(128, 256,
                                  kernel_size=(3, 3, 3),
                                  stride=(2, 1, 1),
                                  padding=(1, 1, 1))
        self.block4 = Conv3dBlock(256, 128,
                                  kernel_size=(3, 3, 3),
                                  stride=(3, 1, 1),
                                  padding=(1, 1, 1))

        self.output_head = nn.Sequential(
            nn.Conv2d(128, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, n_quantiles, kernel_size=1),
        )

    def forward(self, x_seq, sort_quantiles=False, pbar=None):
        """
        Args:
            x_seq         : (B, T, C, H, W) — same format as ConvLSTM
            sort_quantiles: sort outputs at inference to prevent crossing
            pbar          : optional tqdm bar updated per block
        Returns:
            out : (B, 9, H, W)
        """
        # (B, T, C, H, W) → (B, C, T, H, W) for Conv3d
        x = x_seq.permute(0, 2, 1, 3, 4).contiguous()

        x = self.block1(x)
        if pbar is not None: pbar.update(1)

        x = self.block2(x)
        if pbar is not None: pbar.update(1)

        x = self.block3(x)
        if pbar is not None: pbar.update(1)

        x = self.block4(x)
        if pbar is not None: pbar.update(1)

        x   = x.squeeze(2)           # (B, 128, H, W)
        out = self.output_head(x)    # (B, 9,   H, W)

        if sort_quantiles:
            out, _ = torch.sort(out, dim=1)
        return out


# ── Sanity Check ──────────────────────────────────────────────────────────
from convlstm_model import WeightedPinballLoss  # fixed version
from cnn3d_model import CNN3DForecaster
from tqdm import tqdm
import torch

device = torch.device("cpu")
B, T, C, H, W = 2, 12, 21, 140, 131

model     = CNN3DForecaster().to(device)
criterion = WeightedPinballLoss()  # from convlstm_model — has target_clean fix

x      = torch.randn(B, T, C, H, W)
target = torch.rand(B, H, W) * 100
target[0, :5, :5] = float('nan')

with tqdm(total=4, desc="3D-CNN") as pbar:
    pred = model(x, pbar=pbar)

loss = criterion(pred, target)
print(f"Loss: {loss.item():.4f}")
print(f"Pred range: [{pred.min():.3f}, {pred.max():.3f}]")
print(f"NaN in pred: {pred.isnan().any()}")

3D-CNN: 100%|██████████| 4/4 [00:00<00:00, 38.52it/s]

Loss: nan
Pred range: [-0.878, 0.841]
NaN in pred: False


In [5]:
import torch
from convlstm_model import WeightedPinballLoss

criterion = WeightedPinballLoss()

# Use the actual pred and target from above
valid_mask  = ~torch.isnan(target)
target_clean = target.clone()
target_clean[~valid_mask] = 0.0

weights = torch.full_like(target_clean, 1.0)
weights = torch.where(target_clean >= 80,  torch.full_like(weights, 3.0),  weights)
weights = torch.where(target_clean >= 95, torch.full_like(weights, 10.0), weights)

target_exp = target_clean.unsqueeze(1)
taus_exp   = torch.tensor([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]).view(1,-1,1,1)
error      = target_exp - pred

print(f"error NaN: {error.isnan().any()}")
print(f"error range: [{error.min():.2f}, {error.max():.2f}]")

loss = torch.where(error >= 0, taus_exp * error, (taus_exp - 1) * error)
print(f"loss NaN: {loss.isnan().any()}")

loss_mean = loss.mean(dim=1)
print(f"loss_mean NaN: {loss_mean.isnan().any()}")

loss_weighted = loss_mean * weights * valid_mask.float()
print(f"loss_weighted NaN: {loss_weighted.isnan().any()}")

denom = valid_mask.float().sum()
print(f"denom: {denom}")
print(f"final: {loss_weighted.sum() / denom}")

error NaN: False
error range: [-0.49, 100.64]
loss NaN: False
loss_mean NaN: False
loss_weighted NaN: False
denom: 36655.0
final: 59.96180725097656


In [6]:
# TRAIN CONVLSTM — PROBABILISTIC UK FLOOD NOWCASTING
# ============================================================================
# Trains ConvLSTM only. Run train_3dcnn.py separately for 3D-CNN.
# Uses chunked sequential CSV streaming to stay within 8GB RAM.
# ============================================================================
import os
import gc
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from convlstm_model import ConvLSTMForecaster, WeightedPinballLoss

# ── Config ──
CSV_PATH       = "merged_output_hourly_imerg/merged_data_normalised.csv"
CHECKPOINT_DIR = "/DISSERTATION/model_checkpoints/ConvLSTM"
LOG_DIR        = "/DISSERTATION/model_logs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

N_LAT          = 140
N_LON          = 131
CELLS_PER_HOUR = N_LAT * N_LON
N_FEATURES     = 20
N_QUANTILES    = 9
SEQ_LEN        = 12
PATCH_SIZE     = 32
CHUNK_HOURS    = 1500

TRAIN_START    = 0
TRAIN_END      = 15335
VAL_START      = 15336
VAL_END        = 17543

BATCH_SIZE     = 4
N_EPOCHS       = 20
LR             = 1e-4
PATIENCE       = 5
SAVE_EVERY     = 5
TARGET_COL     = "flood_composite_6h_ahead"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    BATCH_SIZE = 16

header_cols  = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
SKIP_COLS    = {"time", "latitude", "longitude", TARGET_COL}
FEATURE_COLS = [c for c in header_cols if c not in SKIP_COLS]
assert len(FEATURE_COLS) == N_FEATURES
print(f"Features: {len(FEATURE_COLS)}")

# ── Land cell index for valid patch sampling ──
print("Building land cell index from first hour...")
first_hour = pd.read_csv(CSV_PATH, nrows=CELLS_PER_HOUR,
                          usecols=["is_land", TARGET_COL])
land_mask  = first_hour["is_land"].to_numpy().reshape(N_LAT, N_LON)
target_hour = first_hour[TARGET_COL].to_numpy().reshape(N_LAT, N_LON)
HALF = PATCH_SIZE // 2

# Valid patch centres: land cells with valid target, away from boundary
valid_centres = [
    (lat, lon)
    for lat in range(HALF, N_LAT - HALF)
    for lon in range(HALF, N_LON - HALF)
    if land_mask[lat, lon] > 0.5
]
print(f"Valid patch centres: {len(valid_centres):,}")


# ── Patch dataset ─────────────────────────────────────────────────────────
class ChunkPatchDataset(Dataset):
    def __init__(self, X_chunk, y_chunk, centres):
        self.X       = X_chunk
        self.y       = y_chunk
        self.centres = centres
        self.seq_ends = list(range(SEQ_LEN, X_chunk.shape[0]))
        # Build index: seq_end × centre combinations (subsample for speed)
        # Use every 10th centre to keep dataset size manageable
        self.centres_sub = centres[::10]
        self.index = [(s, c) for s in self.seq_ends
                      for c in range(len(self.centres_sub))]

    def __len__(self):
        return len(self.index)

    def _patch(self, grid, lat_c, lon_c):
        h  = HALF
        r0 = lat_c - h; r1 = lat_c + h
        c0 = lon_c - h; c1 = lon_c + h
        return grid[..., r0:r1, c0:c1].copy()

    def __getitem__(self, idx):
        seq_end, centre_idx = self.index[idx]
        lat_c, lon_c = self.centres_sub[centre_idx]
        x_seq = np.stack([self._patch(self.X[h], lat_c, lon_c)
                          for h in range(seq_end - SEQ_LEN, seq_end)])
        y = self._patch(self.y[seq_end][np.newaxis], lat_c, lon_c)[0]
        return (torch.from_numpy(x_seq).float(),
                torch.from_numpy(y).float())


# ── Sequential scan epoch ─────────────────────────────────────────────────
def run_epoch(model, criterion, optimizer,
              hour_start, hour_end, is_train, epoch_num):
    mode = "Train" if is_train else "Val  "
    total_loss = 0.0
    n_batches  = 0
    buf_X, buf_y = [], []
    hour_idx = 0

    with tqdm(total=hour_end - hour_start + 1,
              desc=f"  {mode} ep{epoch_num:2d}",
              unit="hr", leave=False,
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} hrs "
                         "[{elapsed}<{remaining}, {rate_fmt}] "
                         "{postfix}") as pbar:

        reader = pd.read_csv(CSV_PATH,
        chunksize=CELLS_PER_HOUR,
        usecols=FEATURE_COLS + [TARGET_COL],
        on_bad_lines="skip",
        low_memory=False,
        dtype={col: 'float32' for col in FEATURE_COLS + [TARGET_COL]}
        )

        for chunk in reader:
            if len(chunk) != CELLS_PER_HOUR:
                hour_idx += 1
                continue
            if hour_idx < hour_start:
                hour_idx += 1
                continue
            if hour_idx > hour_end:
                break

            feat = chunk[FEATURE_COLS].to_numpy(
                dtype=np.float32).reshape(N_LAT, N_LON, N_FEATURES)
            tgt  = chunk[TARGET_COL].to_numpy(
                dtype=np.float32).reshape(N_LAT, N_LON)
            buf_X.append(feat.transpose(2, 0, 1))
            buf_y.append(tgt)
            hour_idx += 1
            pbar.update(1)

            if len(buf_X) == CHUNK_HOURS or hour_idx > hour_end:
                X_chunk = np.stack(buf_X)
                y_chunk = np.stack(buf_y)
                buf_X, buf_y = [], []

                dataset = ChunkPatchDataset(
                    X_chunk, y_chunk, valid_centres)
                if len(dataset) == 0:
                    del X_chunk, y_chunk, dataset
                    gc.collect()
                    continue

                loader = DataLoader(
                    dataset, batch_size=BATCH_SIZE,
                    shuffle=is_train, num_workers=0,
                    drop_last=is_train)

                if is_train:
                    model.train()
                    ctx = torch.enable_grad()
                else:
                    model.eval()
                    ctx = torch.no_grad()

                chunk_loss = 0.0
                n_chunk    = 0
                with ctx:
                    for x, y in loader:
                        x, y = x.to(device), y.to(device)
                        if is_train:
                            optimizer.zero_grad()
                        pred = model(x, sort_quantiles=not is_train)
                        loss = criterion(pred, y)
                        if torch.isnan(loss):
                            continue
                        if is_train:
                            loss.backward()
                            torch.nn.utils.clip_grad_norm_(
                                model.parameters(), 1.0)
                            optimizer.step()
                        chunk_loss += loss.item()
                        n_chunk    += 1

                if n_chunk > 0:
                    avg = chunk_loss / n_chunk
                    total_loss += chunk_loss
                    n_batches  += n_chunk
                    pbar.set_postfix(
                        loss=f"{avg:.4f}", refresh=False)

                del X_chunk, y_chunk, dataset, loader
                gc.collect()

    return total_loss / max(n_batches, 1)


# ── Main training loop ────────────────────────────────────────────────────
model = ConvLSTMForecaster(
    in_channels=N_FEATURES,
    hidden_dims=(64, 128, 64),
    n_quantiles=N_QUANTILES).to(device)
print(f"\nConvLSTM parameters: "
      f"{sum(p.numel() for p in model.parameters()):,}")

optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3)
criterion  = WeightedPinballLoss()

best_val   = np.inf
patience_c = 0
log        = []
t_start    = time.time()

print(f"\nStarting ConvLSTM training — {N_EPOCHS} epochs")
print(f"Checkpoints → {CHECKPOINT_DIR}")

with tqdm(total=N_EPOCHS, desc="ConvLSTM", unit="epoch",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} epochs "
                     "[{elapsed}<{remaining}] {postfix}") as ebar:

    for epoch in range(N_EPOCHS):
        t_ep = time.time()

        avg_train = run_epoch(model, criterion, optimizer,
                              TRAIN_START, TRAIN_END,
                              is_train=True, epoch_num=epoch+1)
        avg_val   = run_epoch(model, criterion, optimizer,
                              VAL_START, VAL_END,
                              is_train=False, epoch_num=epoch+1)

        epoch_min = (time.time() - t_ep) / 60
        elapsed   = (time.time() - t_start) / 60
        scheduler.step(avg_val)

        log.append({"epoch": epoch+1, "train_loss": avg_train,
                    "val_loss": avg_val, "epoch_min": epoch_min})

        ebar.set_postfix(
            train=f"{avg_train:.4f}", val=f"{avg_val:.4f}",
            best=f"{best_val:.4f}", ep_min=f"{epoch_min:.0f}min")
        ebar.update(1)

        if avg_val < best_val:
            best_val   = avg_val
            patience_c = 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_loss": avg_val,
            }, os.path.join(CHECKPOINT_DIR, "best.pt"))
            tqdm.write(f"  ✓ Best val: {avg_val:.4f} → best.pt")
        else:
            patience_c += 1
            if patience_c >= PATIENCE:
                tqdm.write(f"⚠ Early stop ep{epoch+1}")
                break

        if (epoch+1) % SAVE_EVERY == 0:
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "val_loss": avg_val,
            }, os.path.join(CHECKPOINT_DIR, f"epoch_{epoch+1:04d}.pt"))

pd.DataFrame(log).to_csv(
    os.path.join(LOG_DIR, "convlstm_log.csv"), index=False)
print(f"\n✓ ConvLSTM training complete")
print(f"  Best val loss: {best_val:.4f}")
print(f"  Log: {LOG_DIR}/convlstm_log.csv")
print(f"  Best checkpoint: {CHECKPOINT_DIR}/best.pt")

Device: cpu
Features: 20
Building land cell index from first hour...
Valid patch centres: 3,733

ConvLSTM parameters: 1,540,425

Starting ConvLSTM training — 20 epochs
Checkpoints → /Volumes/Seagate_2TB/model_checkpoints/ConvLSTM


ConvLSTM:   0%|          | 0/20 epochs [01:31<?] 


KeyboardInterrupt: 

In [ ]:

import torch
import torch.nn as nn
from tqdm import tqdm

class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size, padding=padding, bias=True)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch_size, height, width, device):
        return (
            torch.zeros(batch_size, self.hidden_channels, height, width, device=device),
            torch.zeros(batch_size, self.hidden_channels, height, width, device=device),
        )

class ConvLSTMLayer(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.cell = ConvLSTMCell(in_channels, hidden_channels, kernel_size)

    def forward(self, x_seq, pbar=None):
        B, T, C, H, W = x_seq.shape
        device = x_seq.device
        h, c = self.cell.init_hidden(B, H, W, device)
        for t in range(T):
            h, c = self.cell(x_seq[:, t], h, c)
            if pbar is not None:
                pbar.update(1)
        return h

class ConvLSTMForecaster(nn.Module):
    def __init__(self, in_channels=21, hidden_dims=(64, 128, 64),
                 n_quantiles=9, kernel_size=3):
        super().__init__()
        self.n_quantiles = n_quantiles
        self.layer1 = ConvLSTMLayer(in_channels,    hidden_dims[0], kernel_size)
        self.layer2 = ConvLSTMLayer(hidden_dims[0], hidden_dims[1], kernel_size)
        self.layer3 = ConvLSTMLayer(hidden_dims[1], hidden_dims[2], kernel_size)
        self.output_head = nn.Sequential(
            nn.Conv2d(hidden_dims[2], 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, n_quantiles, kernel_size=1),
        )

    def forward(self, x_seq, sort_quantiles=False, pbar=None):
        T = x_seq.shape[1]
        h1     = self.layer1(x_seq, pbar=pbar)
        h1_seq = h1.unsqueeze(1).expand(-1, T, -1, -1, -1)
        h2     = self.layer2(h1_seq, pbar=pbar)
        h2_seq = h2.unsqueeze(1).expand(-1, T, -1, -1, -1)
        h3     = self.layer3(h2_seq, pbar=pbar)
        out    = self.output_head(h3)
        if sort_quantiles:
            out, _ = torch.sort(out, dim=1)
        return out

class WeightedPinballLoss(nn.Module):
    def __init__(self, quantile_levels=None,
                 weight_heavy=10.0, weight_avg=3.0, weight_none=1.0,
                 threshold_heavy=95.0, threshold_avg=80.0):
        super().__init__()
        if quantile_levels is None:
            quantile_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
        self.register_buffer("taus",
            torch.tensor(quantile_levels, dtype=torch.float32))
        self.weight_heavy    = weight_heavy
        self.weight_avg      = weight_avg
        self.weight_none     = weight_none
        self.threshold_heavy = threshold_heavy
        self.threshold_avg   = threshold_avg

    def forward(self, pred, target):
        valid_mask = ~torch.isnan(target)
        if valid_mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True, device=pred.device)
        weights = torch.full_like(target, self.weight_none)
        weights = torch.where(target >= self.threshold_avg,
                              torch.full_like(weights, self.weight_avg), weights)
        weights = torch.where(target >= self.threshold_heavy,
                              torch.full_like(weights, self.weight_heavy), weights)
        target_exp = target.unsqueeze(1)
        taus_exp   = self.taus.to(pred.device).view(1, -1, 1, 1)
        error      = target_exp - pred
        loss       = torch.where(error >= 0,
                                 taus_exp * error,
                                 (taus_exp - 1) * error)
        loss_mean     = loss.mean(dim=1)
        loss_weighted = loss_mean * weights * valid_mask.float()
        return loss_weighted.sum() / valid_mask.float().sum()


In [20]:
# TRAIN 3D-CNN — PROBABILISTIC UK FLOOD NOWCASTING
# ============================================================================
# Trains 3D-CNN only. Run train_convlstm.py separately for ConvLSTM.
# Uses chunked sequential CSV streaming to stay within 8GB RAM.
# Identical data loading and evaluation to train_convlstm.py for fair
# architecture comparison.
# ============================================================================
import os
import gc
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from convlstm_model import WeightedPinballLoss
from cnn3d_model import CNN3DForecaster

# ── Config ──
CSV_PATH       = "merged_output_hourly_imerg/merged_data_normalised.csv"
CHECKPOINT_DIR = "/Volumes/Seagate_2TB/model_checkpoints/CNN3D"
LOG_DIR        = "/Volumes/Seagate_2TB/model_logs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

N_LAT          = 140
N_LON          = 131
CELLS_PER_HOUR = N_LAT * N_LON
N_FEATURES     = 20
N_QUANTILES    = 9
SEQ_LEN        = 12
PATCH_SIZE     = 32
CHUNK_HOURS    = 500

TRAIN_START    = 0
TRAIN_END      = 15335
VAL_START      = 15336
VAL_END        = 17543

BATCH_SIZE     = 4
N_EPOCHS       = 20
LR             = 1e-4
PATIENCE       = 5
SAVE_EVERY     = 5
TARGET_COL     = "flood_composite_6h_ahead"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    BATCH_SIZE = 16

header_cols  = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
SKIP_COLS    = {"time", "latitude", "longitude", TARGET_COL}
FEATURE_COLS = [c for c in header_cols if c not in SKIP_COLS]
assert len(FEATURE_COLS) == N_FEATURES
print(f"Features: {len(FEATURE_COLS)}")

# ── Land cell index ───────────────────────────────────────────────────────
print("Building land cell index...")
first_hour = pd.read_csv(CSV_PATH, nrows=CELLS_PER_HOUR,
                          usecols=["is_land", TARGET_COL])
land_mask  = first_hour["is_land"].to_numpy().reshape(N_LAT, N_LON)
HALF = PATCH_SIZE // 2

valid_centres = [
    (lat, lon)
    for lat in range(HALF, N_LAT - HALF)
    for lon in range(HALF, N_LON - HALF)
    if land_mask[lat, lon] > 0.5
]
print(f"Valid patch centres: {len(valid_centres):,}")


# ── Patch dataset ─────────────────────────────────────────────────────────
class ChunkPatchDataset(Dataset):
    def __init__(self, X_chunk, y_chunk, centres):
        self.X           = X_chunk
        self.y           = y_chunk
        self.centres_sub = centres[::10]
        self.seq_ends    = list(range(SEQ_LEN, X_chunk.shape[0]))
        self.index       = [(s, c) for s in self.seq_ends
                            for c in range(len(self.centres_sub))]

    def __len__(self):
        return len(self.index)

    def _patch(self, grid, lat_c, lon_c):
        h  = HALF
        return grid[..., lat_c-h:lat_c+h,
                    lon_c-h:lon_c+h].copy()

    def __getitem__(self, idx):
        seq_end, centre_idx = self.index[idx]
        lat_c, lon_c = self.centres_sub[centre_idx]
        x_seq = np.stack([self._patch(self.X[h], lat_c, lon_c)
                          for h in range(seq_end - SEQ_LEN, seq_end)])
        y = self._patch(self.y[seq_end][np.newaxis], lat_c, lon_c)[0]
        return (torch.from_numpy(x_seq).float(),
                torch.from_numpy(y).float())


# ── Sequential scan epoch ─────────────────────────────────────────────────
def run_epoch(model, criterion, optimizer,
              hour_start, hour_end, is_train, epoch_num):
    mode = "Train" if is_train else "Val  "
    total_loss = 0.0
    n_batches  = 0
    buf_X, buf_y = [], []
    hour_idx = 0

    with tqdm(total=hour_end - hour_start + 1,
              desc=f"  {mode} ep{epoch_num:2d}",
              unit="hr", leave=False,
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} hrs "
                         "[{elapsed}<{remaining}, {rate_fmt}] "
                         "{postfix}") as pbar:

        reader = pd.read_csv(
            CSV_PATH, chunksize=CELLS_PER_HOUR,
            usecols=FEATURE_COLS + [TARGET_COL],
            on_bad_lines="skip", low_memory=False)

        for chunk in reader:
            if len(chunk) != CELLS_PER_HOUR:
                hour_idx += 1
                continue
            if hour_idx < hour_start:
                hour_idx += 1
                continue
            if hour_idx > hour_end:
                break

            feat = chunk[FEATURE_COLS].to_numpy(
                dtype=np.float32).reshape(N_LAT, N_LON, N_FEATURES)
            tgt  = chunk[TARGET_COL].to_numpy(
                dtype=np.float32).reshape(N_LAT, N_LON)
            buf_X.append(feat.transpose(2, 0, 1))
            buf_y.append(tgt)
            hour_idx += 1
            pbar.update(1)

            if len(buf_X) == CHUNK_HOURS or hour_idx > hour_end:
                X_chunk = np.stack(buf_X)
                y_chunk = np.stack(buf_y)
                buf_X, buf_y = [], []

                dataset = ChunkPatchDataset(
                    X_chunk, y_chunk, valid_centres)
                if len(dataset) == 0:
                    del X_chunk, y_chunk, dataset
                    gc.collect()
                    continue

                loader = DataLoader(
                    dataset, batch_size=BATCH_SIZE,
                    shuffle=is_train, num_workers=0,
                    drop_last=is_train)

                if is_train:
                    model.train()
                    ctx = torch.enable_grad()
                else:
                    model.eval()
                    ctx = torch.no_grad()

                chunk_loss = 0.0
                n_chunk    = 0
                with ctx:
                    for x, y in loader:
                        x, y = x.to(device), y.to(device)
                        if is_train:
                            optimizer.zero_grad()
                        pred = model(x, sort_quantiles=not is_train)
                        loss = criterion(pred, y)
                        if torch.isnan(loss):
                            continue
                        if is_train:
                            loss.backward()
                            torch.nn.utils.clip_grad_norm_(
                                model.parameters(), 1.0)
                            optimizer.step()
                        chunk_loss += loss.item()
                        n_chunk    += 1

                if n_chunk > 0:
                    avg = chunk_loss / n_chunk
                    total_loss += chunk_loss
                    n_batches  += n_chunk
                    pbar.set_postfix(
                        loss=f"{avg:.4f}", refresh=False)

                del X_chunk, y_chunk, dataset, loader
                gc.collect()

    return total_loss / max(n_batches, 1)


# ── Main training loop ────────────────────────────────────────────────────
model = CNN3DForecaster(
    in_channels=N_FEATURES,
    n_quantiles=N_QUANTILES).to(device)
print(f"\n3D-CNN parameters: "
      f"{sum(p.numel() for p in model.parameters()):,}")

optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3)
criterion  = WeightedPinballLoss()

best_val   = np.inf
patience_c = 0
log        = []
t_start    = time.time()

print(f"\nStarting 3D-CNN training — {N_EPOCHS} epochs")
print(f"Checkpoints → {CHECKPOINT_DIR}")

with tqdm(total=N_EPOCHS, desc="CNN3D", unit="epoch",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} epochs "
                     "[{elapsed}<{remaining}] {postfix}") as ebar:

    for epoch in range(N_EPOCHS):
        t_ep = time.time()

        avg_train = run_epoch(model, criterion, optimizer,
                              TRAIN_START, TRAIN_END,
                              is_train=True, epoch_num=epoch+1)
        avg_val   = run_epoch(model, criterion, optimizer,
                              VAL_START, VAL_END,
                              is_train=False, epoch_num=epoch+1)

        epoch_min = (time.time() - t_ep) / 60
        elapsed   = (time.time() - t_start) / 60
        scheduler.step(avg_val)

        log.append({"epoch": epoch+1, "train_loss": avg_train,
                    "val_loss": avg_val, "epoch_min": epoch_min})

        ebar.set_postfix(
            train=f"{avg_train:.4f}", val=f"{avg_val:.4f}",
            best=f"{best_val:.4f}", ep_min=f"{epoch_min:.0f}min")
        ebar.update(1)

        if avg_val < best_val:
            best_val   = avg_val
            patience_c = 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_loss": avg_val,
            }, os.path.join(CHECKPOINT_DIR, "best.pt"))
            tqdm.write(f"  ✓ Best val: {avg_val:.4f} → best.pt")
        else:
            patience_c += 1
            if patience_c >= PATIENCE:
                tqdm.write(f"⚠ Early stop ep{epoch+1}")
                break

        if (epoch+1) % SAVE_EVERY == 0:
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "val_loss": avg_val,
            }, os.path.join(CHECKPOINT_DIR, f"epoch_{epoch+1:04d}.pt"))

pd.DataFrame(log).to_csv(
    os.path.join(LOG_DIR, "cnn3d_log.csv"), index=False)
print(f"\n✓ 3D-CNN training complete")
print(f"  Best val loss: {best_val:.4f}")
print(f"  Log: {LOG_DIR}/cnn3d_log.csv")
print(f"  Best checkpoint: {CHECKPOINT_DIR}/best.pt")

Device: cuda
Features: 20
Building land cell index...
Valid patch centres: 3,733

3D-CNN parameters: 534,281

Starting 3D-CNN training — 20 epochs
Checkpoints → /Volumes/Seagate_2TB/model_checkpoints/CNN3D


CNN3D:   0%|          | 0/20 epochs [00:12<?] 


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!